In [1]:
import torch # Torch Module
import torch.nn as nn # Neural Network

In [2]:
"""
Why "tensor" and not "array" — the actual reason
NumPy arrays don't track gradients or operation history at all — they're purely for numerical computation. 
A PyTorch tensor is functionally a NumPy array that additionally knows how to record its own computational history and 
compute gradients through that history automatically. That gradient-tracking capability is the entire reason PyTorch tensors exist as 
a distinct concept, rather than just using NumPy directly for everything
"""

'\nWhy "tensor" and not "array" — the actual reason\nNumPy arrays don\'t track gradients or operation history at all — they\'re purely for numerical computation. \nA PyTorch tensor is functionally a NumPy array that additionally knows how to record its own computational history and \ncompute gradients through that history automatically. That gradient-tracking capability is the entire reason PyTorch tensors exist as \na distinct concept, rather than just using NumPy directly for everything\n'

In [3]:
# We explicity add a dedcimal because we want the numeric values to be floating numbers and not integers.
# This matters because weights, gradients, and most neural network math need floats, not integers.
x = torch.tensor(
                  [[2., 1., 0., 3.],
                   [1., 0., 2., 1.],
                   [3., 2., 1., 0.],
                   [0., 1., 3., 2.]]
)

In [4]:
""" 
    If shapes don't align for an operation, it throws an error immediately, rather than silently computing something wrong.
    Checking .shape after every operation, especially while learning, is not optional busywork — 
    it's how you verify your mental model of what just happened matches what actually happened.
"""

" \n    If shapes don't align for an operation, it throws an error immediately, rather than silently computing something wrong.\n    Checking .shape after every operation, especially while learning, is not optional busywork — \n    it's how you verify your mental model of what just happened matches what actually happened.\n"

In [5]:
print(x.shape)      # torch.Size([4, 4])
print(x.shape[0])   # 4  — number of rows
print(x.dim())       # 2  — number of dimensions

torch.Size([4, 4])
4
2


In [6]:
# Basic Operations : 

a = torch.tensor([1., 2., 3.])
b = torch.tensor([4., 5., 6.])

print(a + b)        # tensor([5., 7., 9.])   — element-wise addition 
print(a * b)        # tensor([4., 10., 18.]) — element-wise multiplication, NOT dot product ( Hadamard Product )
print(torch.dot(a, b))  # tensor(32.)         — actual dot product: 1×4+2×5+3×6=32

tensor([5., 7., 9.])
tensor([ 4., 10., 18.])
tensor(32.)


In [7]:
# Matrix multiplication —  z = W·x calculation
W = torch.tensor([[0.5, -1.0]])   # shape (1, 2) — our FC layer's weight from the CNN exercise
flat = torch.tensor([[3.0, 1.0]]) # shape (1, 2) — our flattened pooled values

# Matrix multiply: (1,2) can't directly multiply (1,2) — need to transpose
z = flat @ W.T + 0.2
print(z)   # tensor([[0.7000]])

tensor([[0.7000]])


In [8]:
# Autograd - computes partial derivatives for a given tensor automatically if any size.
w = torch.tensor(2.0, requires_grad=True)
x_val = torch.tensor(3.0)

z = w * x_val # The partial derivative : ∂z/∂w = x_val = 3

z.backward()
print(w.grad)   # tensor(3.)

tensor(3.)


In [9]:
# if z = w² + 3w and w = 2, what should dz/dw be at that point (compute the derivative expression first, then plug in w=2)

w = torch.tensor(2.0, requires_grad=True)
z = w**2 + 3*w
z.backward()

print(w.grad)   # tensor(7.)

tensor(7.)


In [12]:
# Building an nn.Module
class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        # nn.Linear(#inputs,#outputs) for that layer
        self.layer1 = nn.Linear(4, 3)   # W¹ is (3,4), matches your hand-derived shape rule
        self.layer2 = nn.Linear(3, 2)   # W² is (2,3)
        self.layer3 = nn.Linear(2, 1)   # W³ is (1,2)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # Forward Traversing and calculating each layer's output 
        # a0 = X itself
        a1 = self.sigmoid(self.layer1(x))   # a¹ = σ(W¹x + b¹)
        a2 = self.sigmoid(self.layer2(a1))  # a² = σ(W²a¹ + b²)
        y_hat = self.sigmoid(self.layer3(a2))  # ŷ = σ(W³a² + b³)
        return y_hat

In [ ]:
model = SimpleNet() # Instantiate the model object

x = torch.tensor([[1.0, 2.0, 3.0, 4.0]])   # shape (1, 4) — batch of 1, 4 features
output = model(x)   # NOT model.forward(x) 
print(output)        # some tensor, shape (1, 1), a random value since weights are randomly initialized

tensor([[0.6172]], grad_fn=<SigmoidBackward0>)


In [ ]:
"""
Why model(x) and not model.forward(x) directly — this is a specific PyTorch convention worth locking in now. 
nn.Module overrides Python's __call__ method, and that override does some important bookkeeping (like hooks for training mode, gradient tracking setup) 
around your forward method. Calling model.forward(x) directly skips that bookkeeping. Always call model(x), 
never model.forward(x), even though both will often appear to produce the same output in simple cases
"""

In [19]:
# Let's inspect the model parameters
for name, param in model.named_parameters():
    print(f'name: {name}\t param_shape: {param.shape}')

name: layer1.weight	 param_shape: torch.Size([3, 4])
name: layer1.bias	 param_shape: torch.Size([3])
name: layer2.weight	 param_shape: torch.Size([2, 3])
name: layer2.bias	 param_shape: torch.Size([2])
name: layer3.weight	 param_shape: torch.Size([1, 2])
name: layer3.bias	 param_shape: torch.Size([1])


In [20]:
# This we did was jut a forward pass only using forward method.

In [21]:
# Another example of this

In [25]:
class SimpleNet (nn.Module):
    def __init__(self):
        super().__init__()
        # Creating the neural layers
        self.layer1 = nn.Linear(4,3)
        self.layer2 = nn.Linear(3,2)
        self.layer3 = nn.Linear(2,1)
        self.sigmoid = nn.Sigmoid()

    def forward(self,X):
        # Forward Traversal and their outputs
        a1 = self.sigmoid(self.layer1(X))
        a2 = self.sigmoid(self.layer2(a1))
        y_hat = self.sigmoid(self.layer3(a2))
        
        return y_hat
    
model = SimpleNet()
X = torch.tensor([[1.0, 2.0, 3.0, 4.0]])

print(f' y_hat : {model(X)} \n with shape : {model(X).shape} ')

# OP : 
# y_hat : tensor([[0.4233]], grad_fn=<SigmoidBackward0>) 
# with shape : torch.Size([1, 1]) 

 y_hat : tensor([[0.4658]], grad_fn=<SigmoidBackward0>) 
 with shape : torch.Size([1, 1]) 
